# Stage 1 — 실험 1 (확증)

**하려는 것:** 앞선 코드에 규칙 어긴 예시가 많아질수록, 모델이 새로 만드는 함수도 규칙을 덜 지키는가?

- **조작:** 새 함수를 시키기 전 앞에 깔 코드 12개 중 **규칙 지킨(camel) 예시를 4/3/2/1/0개** 남긴다.
- **측정:** 그 직후 만든 **첫 함수가 camelCase를 지켰는지**.
- **기대:** 지킨 예시가 줄수록 준수율↓ (파일럿 100→75→62→38→0%).

**GPU 런타임 필요.** 런타임 → 런타임 유형 변경 → T4 GPU.

> 결과는 Google Drive에 저장한다(아래 2번). VM이 끊겨도 안 사라지고, 실행 셀을 다시 돌리면 이어서 된다.

## 0. 최신 코드 받기

In [ ]:
import os
REPO_URL = "https://github.com/deanjs/instruction-adherence.git"
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
# 원격 최신으로 강제 동기화 (추적 안 되는 결과 파일은 안 지워짐)
!git fetch origin -q && git reset --hard origin/main
!git log --oneline -1

## 1. GPU · 의존성 확인

In [ ]:
!nvidia-smi -L
!pip install -q "transformers>=4.51.0" "accelerate>=0.26.0"
import torch, transformers
print("transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available())

## 2. Drive에 결과 저장 준비 (VM 재활용 대비)

처음 실행 시 Drive 권한 승인 팝업이 뜬다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/instruction-adherence/exp1_main.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
print("저장 위치:", OUT)

## 3. 실행

조건 5(compliant 4/3/2/1/0) × seed 20 × 함수 3 = 300 생성. T4에서 대략 30~40분.

- **끊기면 이 셀만 다시 실행** → 완료분 건너뛰고 이어감.
- GPU 아끼려면 먼저 `--n-seeds 10`으로 돌려보고, 괜찮으면 20으로 늘려도 됨(재개되니 손해 없음).

In [ ]:
!python src/exp1_main.py --n-seeds 20 --chain --out "$OUT"

## 4. 요약만 다시 보기 (생성 없이, 언제든)

**봐야 할 것:** compliant 4→0으로 갈수록 준수율이 매끄럽게 떨어지는지(파일럿 100→75→62→38→0 재현), H1a 대비 차이가 큰지. 그리고 연쇄효과(위치1 O/X → 위치2·3).

In [ ]:
!python src/exp1_main.py --summary-only --chain --out "$OUT"

## 5. (선택) 결과 내려받기

Drive에 이미 있으니 보통 필요 없지만, 로컬 repo에 커밋해 보존하려면:

In [ ]:
from google.colab import files
files.download(OUT)